In [4]:
pip install opencv-contrib-python


   ---------------------------------------- 0.0/45.3 MB ? eta -:--:--
    --------------------------------------- 1.0/45.3 MB 7.6 MB/s eta 0:00:06
   -- ------------------------------------- 2.4/45.3 MB 6.2 MB/s eta 0:00:07
   --- ------------------------------------ 3.7/45.3 MB 6.3 MB/s eta 0:00:07
   ---- ----------------------------------- 5.2/45.3 MB 6.7 MB/s eta 0:00:06
   ----- ---------------------------------- 6.0/45.3 MB 6.7 MB/s eta 0:00:06
   ------ --------------------------------- 6.8/45.3 MB 5.5 MB/s eta 0:00:08
   ------ --------------------------------- 7.6/45.3 MB 5.3 MB/s eta 0:00:08
   -------- ------------------------------- 9.2/45.3 MB 5.4 MB/s eta 0:00:07
   --------- ------------------------------ 10.2/45.3 MB 5.4 MB/s eta 0:00:07
   ---------- ----------------------------- 11.8/45.3 MB 5.6 MB/s eta 0:00:06
   ----------- ---------------------------- 13.1/45.3 MB 5.6 MB/s eta 0:00:06
   ------------ --------------------------- 14.7/45.3 MB 5.8 MB/s eta 0:00:06
  

SVM untuk voice.csv

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# =======================
# LOAD DATA
# =======================
df = pd.read_csv("voice.csv")

# Encode label male/female → 0/1
df["label"] = df["label"].map({"male": 0, "female": 1})

X = df.drop("label", axis=1)
y = df["label"]

# =======================
# FUNGSI TRAIN SVM
# =======================
def run_svm(X, y, test_size, kernel):
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    # Normalisasi
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Model SVM
    model = SVC(kernel=kernel)
    model.fit(X_train, y_train)

    # Prediksi
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)

    return acc

# =======================
# MENJALANKAN SEMUA MODEL
# =======================
kernels = ["linear", "poly", "rbf"]
splits = [0.3, 0.2]

results = []

for s in splits:
    for k in kernels:
        acc = run_svm(X, y, s, k)
        results.append([f"{int((1-s)*100)}:{int(s*100)}", k, acc])

# TABEL HASIL
df_results = pd.DataFrame(results, columns=["Split", "Kernel", "Akurasi"])
print("\n=== HASIL SVM VOICE DATASET ===")
print(df_results)



=== HASIL SVM VOICE DATASET ===
   Split  Kernel   Akurasi
0  70:30  linear  0.970557
1  70:30    poly  0.957939
2  70:30     rbf  0.981073
3  80:20  linear  0.976341
4  80:20    poly  0.971609
5  80:20     rbf  0.982650


Klasifikasi Siang/Malam

In [12]:
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np

# ========================================
# 1. PREPROCESS (RESIZE + GRAYSCALE + HOG)
# ========================================
def extract_hog_features(images):
    hog_features = []

    for img in images:
        # Resize biar konsisten
        img_resized = cv2.resize(img, (128, 128))

        # Grayscale
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

        # Ekstraksi HOG
        features = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys'
        )

        hog_features.append(features)

    return np.array(hog_features)


print("Ekstraksi HOG untuk training...")
X_train = extract_hog_features(train_images)
y_train = np.array(train_labels)

print("Ekstraksi HOG untuk testing...")
X_test = extract_hog_features(test_images)
y_test = np.array(test_labels)

print("Shape fitur train:", X_train.shape)
print("Shape fitur test :", X_test.shape)

# ============================
# 2. TRAINING MODEL SVM
# ============================
print("Training model SVM...")
model = SVC(kernel="rbf", C=10, gamma=0.001)
model.fit(X_train, y_train)

# ============================
# 3. EVALUASI MODEL
# ============================
print("Evaluasi pada test set...")
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Akurasi Test:", acc * 100, "%")

# ============================
# 4. CONTOH PREDIKSI SATU GAMBAR
# ============================
def predict_single_image(img):
    img_resized = cv2.resize(img, (128, 128))
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

    feat = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )

    feat = feat.reshape(1, -1)
    pred = model.predict(feat)[0]

    return "day" if pred == 0 else "night"


# Test prediksi gambar pertama
if len(test_images) > 0:
    print("Contoh prediksi gambar pertama:", predict_single_image(test_images[0]))


Ekstraksi HOG untuk training...
Ekstraksi HOG untuk testing...
Shape fitur train: (240, 8100)
Shape fitur test : (160, 8100)
Training model SVM...
Evaluasi pada test set...
Akurasi Test: 94.375 %
Contoh prediksi gambar pertama: day
